# 第 11 章 物価・失業とフィリップス曲線: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-basics-programming/11-phillips-curve-and-okun.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
unemployment = 4.0        # 失業率 u（%）
natural_rate = 5.0        # 自然失業率 u_n（%）
expected_inflation = 2.0  # 期待インフレ率 π^e（%）
beta = 0.5                # 失業率への感応度 β
periods = 5               # 期待が追いつく経路の期間
growth = [3.2, 1.5, 4.1, 0.8, 2.9, 5.0, -0.5, 3.6, 2.2, 4.5, 1.0, 3.0]  # 成長率（%）
okun_slope = -0.4         # 架空のデータの、真の傾き
okun_intercept = 1.2      # 架空のデータの、真の切片
noise = 0.1               # 架空のデータの、雑音の大きさ
seed = 0                  # 雑音の種
nominal_rate = 1.0        # 名目利子率（%）
inflation = -2.0          # インフレ率（%）。負ならデフレ
debt = 1000               # 名目の借金
years = 10                # 借金の年数
mpc = 0.6                 # 限界消費性向 c
b = 20                    # 投資の利子感応度 b
intercept = 450           # IS 曲線の A
k = 0.5                   # 貨幣需要の所得感応度 k
h = 100                   # 貨幣需要の利子感応度 h
nominal_money = 250       # 名目貨幣供給 M
delta_government = 10     # 需要の増加（ディマンドプル）
supply_loss = 50          # 供給の減少（コストプッシュ）

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))
_loaded_lib = sys.modules.get("lib")
if _loaded_lib is not None and (
    Path(getattr(_loaded_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in tuple(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## フィリップス曲線

In [ ]:
from lib.chapter11.phillips import (
    accelerating_inflation_path,
    implied_natural_rate,
    short_run_inflation,
)

pi = short_run_inflation(unemployment, natural_rate, expected_inflation, beta)
print(f"短期のインフレ率: {pi:.2f}%")
print("失業率を保ち続けたときのインフレ率の経路:",
      accelerating_inflation_path(
          unemployment, natural_rate, expected_inflation, beta, periods
      ))
print("観測から逆算した自然失業率:",
      implied_natural_rate(unemployment, pi, expected_inflation, beta))

In [ ]:
from lib.chapter11.plot import plot_phillips

plot_phillips(
    natural_rate, [expected_inflation, expected_inflation + 2], beta, u_min=3, u_max=7
)

## オークンの法則

In [ ]:
from lib.chapter11.okun import (
    okun_breakeven_growth,
    okun_coefficients,
    synthetic_unemployment_change,
)

change = synthetic_unemployment_change(growth, okun_slope, okun_intercept, noise, seed)
fit = okun_coefficients(growth, change)
print(f"推定: 傾き {fit.slope:.3f}、切片 {fit.intercept:.3f}")
print(f"失業率が変わらない成長率: {okun_breakeven_growth(fit):.3f}%")

In [ ]:
from lib.chapter11.plot import plot_okun

plot_okun(growth, change, fit)

## デフレと、物価ショック

In [ ]:
from lib.chapter10.ad_as import AdEconomy
from lib.chapter11.deflation import debt_burden, real_interest_rate
from lib.chapter11.price_shocks import cost_push_effect, demand_pull_effect

print(f"実質利子率: {real_interest_rate(nominal_rate, inflation):.1f}%")
burden = debt_burden(debt, inflation / 100, years)
print(f"借金の実質的な負担（{years} 年後）: {burden:.1f}")
economy = AdEconomy(mpc, b, intercept, k, h, nominal_money)
dp, dy = demand_pull_effect(economy, 1000, 480, 400, delta_government)
print(f"ディマンドプル: 物価 {dp:+.4f}、所得 {dy:+.2f}")
cp, cy = cost_push_effect(economy, 1000, 480, 400, supply_loss)
print(f"コストプッシュ: 物価 {cp:+.4f}、所得 {cy:+.2f}")

## 試してみよう

- `expected_inflation = 4.0` にすると、短期のインフレ率は 4.50% になります（2.0 のときは
  2.50%）。期待インフレ率が高いと、同じ失業率でも、インフレ率が高くなります
- `beta = 1.0` にすると、短期のインフレ率は 3.00% です。曲線が急になります
- `unemployment = 4.0`（自然失業率の 5.0 より低い）を保ち続けると、経路は
  [2, 2.5, 3.0, 3.5, 4.0, 4.5] と上がり続けます。`unemployment = 5.0` にすると、
  [2, 2.0, 2.0, 2.0, 2.0, 2.0] で、上がりません。**長期のトレードオフはありません**
- `noise = 0.5` にすると、推定の傾きは −0.442、切片は 1.320 になります
  （`noise = 0.1` では −0.408 と 1.224）。雑音が大きいと、推定が真の値（−0.4、1.2）
  から離れます
- `inflation = 2.0` にすると、実質利子率は −1.0%、借金の実質的な負担は 820.3 です
  （`inflation = -2.0` では 3.0% と 1223.9）。デフレは、実質利子率を高くし、
  借金の負担を重くします
- `delta_government = 5` にすると、ディマンドプルは、物価 +0.0216、所得 +8.64 です
  （10 では +0.0433 と +17.32）。`supply_loss = 50` のコストプッシュは、物価 +0.7500、
  所得 −30.00 で、**所得が減ります**